In [ ]:
import pandas as pd
#Seteando tamaño max de columnas y registros
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.options.display.float_format = '{:.2f}'.format

import numpy as np
import re
import csv
import os
import pickle
from glob import glob

import matplotlib.pyplot as plt
import time

### DEBAGREEMENT

In [ ]:
deba = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/DEBAGREEMENT_data/Labeled Dataset/labeled_data.csv",
                   sep=","
                   )
deba#.head(5)

In [ ]:
deba.describe()

In [ ]:
deba['agreement_fraction'].unique()

### Comments by Author (from Reddit)

In [ ]:
authors = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/authors.csv",
                      sep="|"
                      )
authors

### Comments by Author

In [ ]:
deba_auth = pd.merge(deba, authors, how='left', left_on='author_parent', right_on='author')
deba_auth.drop(columns=['author'], inplace=True)
deba_auth.rename(columns={"count": "count_parent"}, inplace=True)
deba_auth

In [ ]:
deba_auth = pd.merge(deba_auth, authors, how='left', left_on='author_child', right_on='author')
deba_auth.drop(columns=['author'], inplace=True)
deba_auth.rename(columns={"count": "count_child"}, inplace=True)
deba_auth

In [ ]:
len(deba_auth['author_parent'].unique())

In [ ]:
len(deba_auth['author_child'].unique())

In [ ]:
1-(12530-7637)/12530 # Parents

In [ ]:
1-(14974-8827)/14974 # Childs

In [ ]:
deba_auth = deba_auth[(~deba_auth['count_parent'].isna())&(~deba_auth['count_child'].isna())&
                      (deba_auth['count_parent']>=100)&(deba_auth['count_child']>=100)]
deba_auth

In [ ]:
len(deba_auth['author_parent'].unique())

In [ ]:
len(deba_auth['author_child'].unique())

In [ ]:
deba_auth[deba_auth['count_parent']>100]

In [ ]:
#deba_auth = deba_auth[(deba_auth['count_parent']>200)&(deba_auth['count_child']>200)]

In [ ]:
unique_authors = pd.unique(deba_auth['author_parent'].tolist() + deba_auth['author_child'].tolist()).tolist()

In [ ]:
len(unique_authors)

In [ ]:
with open("unique_authors.pkl", "wb") as f:
    pickle.dump(unique_authors, f)

In [ ]:
with open("/Proyecto/Value-disagreement/Python/Datasets/unique_authors.pkl", "rb") as f:
    unique_authors = pickle.load(f)

In [ ]:
unique_authors

In [ ]:
len(deba_auth[deba_auth['count_parent']>100]['author_parent'].unique())

In [ ]:
len(deba_auth[deba_auth['count_parent']>100]['author_child'].unique())

In [ ]:
len(deba_auth[deba_auth['count_child']>500]['author_parent'].unique())

In [ ]:
len(deba_auth[deba_auth['count_child']>500]['author_child'].unique())

### JOIN PROFILES

In [ ]:
deba_auth['author_parent'] = deba_auth['author_parent'].str.upper()
deba_auth['author_child'] = deba_auth['author_child'].str.upper()
deba_auth

In [ ]:
df_compact = pd.read_csv("/Proyecto/Value-disagreement/Python/Models/Inference/final_profiles/author_profiles_compact.csv",sep="|")
df_compact['n_comments_all'] = df_compact['n_comments_all']/10
df_compact['author'] = df_compact['author'].str.upper()
df_compact.head(2)

In [ ]:
df_compact = df_compact[['author','n_comments_all','total_value_mentions','diversity_values_expressed','values_order',
                         'top3_binary_values','top3_prob_values','top3_paper_values',
                         'entropy_binary_prev','entropy_mean_prob_profile','entropy_paper_profile',
                         'pos_count_vector','prev_vector','mean_prob_vector','std_prob_vector','paper_profile_vector']]
df_compact

In [ ]:
deba_auth = pd.merge(deba_auth, df_compact, how='left', left_on='author_parent', right_on='author')
deba_auth.drop(columns=['author'], inplace=True)
deba_auth.rename(columns={"values_order":"values_order",
                          "n_comments_all": "p_total_comments",
                          "total_value_mentions":"p_total_value_mentions",
                          "diversity_values_expressed":"p_values_expressed",
                          "top3_binary_values": "p_top3_binary_values",
                          "top3_prob_values": "p_top3_prob_values",
                          "top3_paper_values":"p_top3_paper_values",
                          "entropy_binary_prev": "p_entropy_binary",
                          "entropy_mean_prob_profile":"p_entropy_mean_prob",
                          "entropy_paper_profile":"p_entropy_paper",
                          "pos_count_vector": "p_count_vector",
                          "prev_vector": "p_binary_vector",
                          "mean_prob_vector": "p_prob_vector",
                          "std_prob_vector": "p_std_vector",
                          "paper_profile_vector":"p_paper_vector",
                          }, inplace=True)
deba_auth

In [ ]:
deba_auth = pd.merge(deba_auth, df_compact, how='left', left_on='author_child', right_on='author')
deba_auth.drop(columns=['author'], inplace=True)
deba_auth.rename(columns={"n_comments_all": "c_total_comments",
                          "total_value_mentions":"c_total_value_mentions",
                          "diversity_values_expressed":"c_values_expressed",
                          "top3_binary_values": "c_top3_binary_values",
                          "top3_prob_values": "c_top3_prob_values",
                          "top3_paper_values":"c_top3_paper_values",
                          "entropy_binary_prev": "c_entropy_binary",
                          "entropy_mean_prob_profile":"c_entropy_mean_prob",
                          "entropy_paper_profile":"c_entropy_paper",
                          "pos_count_vector": "c_count_vector",
                          "prev_vector": "c_binary_vector",
                          "mean_prob_vector": "c_prob_vector",
                          "std_prob_vector": "c_std_vector",
                          "paper_profile_vector":"c_paper_vector",                        
                          }, inplace=True)
deba_auth

In [ ]:
deba_auth.drop(columns=['values_order_y'], inplace=True)
deba_auth.rename(columns={"values_order_x": "values_order"}, inplace=True)
deba_auth

In [ ]:
deba_auth.sort_values('agreement_fraction')[['label','agreement_fraction','individual_kappa','count_parent','p_total_comments','count_child','p_total_comments','p_entropy_bp','c_entropy_bp','c_values_amount','p_values_amount']]

In [ ]:
deba_auth

In [ ]:
deba_auth.to_csv(r"/Proyecto/Value-disagreement/Python/Models/Inference/final_profiles/deba_with_profiles.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
              )